In [ ]:
# !git clone https://github.com/Nikhub7-dotcom/MinorProjectV3.git
!ls MinorProjectV3/poison-cifar/results | wc -l

72


In [ ]:
# Cell 1: setup (git, data, preflight). Safe to re-run.
import os, sys, shutil, base64, tarfile, subprocess as sp
from pathlib import Path
from google.colab import userdata
import torch, torchvision, yaml

UP, FORK = "https://github.com/Nikhub7-dotcom/MinorProjectV3.git", "https://github.com/shivamm-verma/MinorProjectV3.git"
BR, REPO, DATA = "shivam/imagenet-results", "/content/MinorProjectV3", "/content/data"
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
AUTH = "http.https://github.com/.extraheader=AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()

def run(cmd, cwd=None, show=True):
    """Run a command and stream its output INTO the notebook (plain subprocess output is invisible in Colab)."""
    p = sp.Popen(cmd, cwd=cwd, stdout=sp.PIPE, stderr=sp.STDOUT, text=True, bufsize=1); out = []
    for l in p.stdout:
        out.append(l); show and print(l, end="")
    return p.wait(), "".join(out)

def g(*a, auth=False, show=False):
    return run(["git", *(["-c", AUTH] if auth else []), *a], cwd=PROJ, show=show)

if not (Path(REPO) / ".git").is_dir():
    shutil.rmtree(REPO, ignore_errors=True)
    assert run(["git", "-c", AUTH, "clone", UP, REPO])[0] == 0
PROJ = next(str(p) for p in (Path(REPO), Path(REPO) / "poison-cifar") if (p / "src" / "run_experiment.py").is_file())
for n, u in (("origin", FORK), ("upstream", UP)):
    g("remote", "add", n, u); g("remote", "set-url", n, u)
g("config", "user.name", "Shivam Verma"); g("config", "user.email", "shivam.verma256@outlook.com")

# Use the local results branch if present; else start from the remote results branch (finished runs get skipped), else from main.
g("reset", "-q", "--hard")
if g("rev-parse", "--verify", "-q", BR)[0] == 0: g("checkout", "-q", BR); base = f"existing local {BR}"   # keep unpushed local commits
else:
    base = None
    for r in ("upstream", "origin"):
        if g("fetch", r, BR, auth=True)[0] == 0: base = f"{r}/{BR}"; break
    if base is None: g("fetch", "upstream", "main", auth=True); base = "upstream/main"
    assert g("checkout", "-q", "-B", BR, base, show=True)[0] == 0
print("base:", base)

# Which remotes can we really push to? (dry run, needs write access)
PUSH_ORDER = [r for r in ("upstream", "origin") if g("push", "--dry-run", r, f"HEAD:refs/heads/{BR}", auth=True)[0] == 0]
print("writable remotes (in push order):", PUSH_ORDER)
assert PUSH_ORDER, "Token cannot push to either repo: give GITHUB_TOKEN 'Contents: read/write' (fine-grained) or 'repo' scope (classic)."

# Dataset (Imagenette 160px, 10 classes)
if not (Path(DATA, "train").is_dir() and Path(DATA, "val").is_dir()):
    os.makedirs(DATA, exist_ok=True)
    run(["wget", "-q", "-O", "/content/im.tgz", "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"])
    tarfile.open("/content/im.tgz").extractall(DATA, filter="data")
    for s in ("train", "val"): shutil.move(f"{DATA}/imagenette2-160/{s}", f"{DATA}/{s}")

# Patched COPY of src (smaller eval batch -> no OOM on DenseNet/VGG). Repo files stay untouched, so git stays clean.
shutil.rmtree("/content/src_run", ignore_errors=True); shutil.copytree(f"{PROJ}/src", "/content/src_run")
run(["sed", "-i", "s/batch_size=256/batch_size=64/g", "/content/src_run/run_experiment.py"])
shutil.rmtree(f"{PROJ}/checkpoints", ignore_errors=True); shutil.rmtree(f"{PROJ}/logs", ignore_errors=True)  # drop stale partial runs
print("GPU:", torch.cuda.get_device_name(0), "| classes:", len(os.listdir(f"{DATA}/train")), "| SETUP OK")

base: existing local shivam/imagenet-results
writable remotes (in push order): ['upstream']
GPU: Tesla T4 | classes: 11 | SETUP OK


In [ ]:
# Cell 2: train all 9 models, push each result the moment it finishes. Re-run to resume/skip finished ones.
import time, yaml, glob
from pathlib import Path
MODELS, RATES = ("resnet50", "vgg19", "densenet161"), ("0_1pct", "0_5pct", "1pct")
AUTO_DISCONNECT = True          # free the GPU when all 9 are done
BS, status, t_all = {}, {}, time.time()   # BS remembers the batch size that fits per model

def push(run_id):
    g("add", "-f", f"results/{run_id}_result.json"); g("commit", "-q", "-m", f"Add {run_id} result")
    for r in PUSH_ORDER:
        for _ in range(3):
            if g("push", "-u", r, BR, auth=True, show=True)[0] == 0: return r
            g("pull", "--rebase", "--autostash", r, BR, auth=True); time.sleep(10)

def train(model, cfg):
    for attempt in range(6):
        bs = BS.setdefault(model, 128); lr = 0.1 * bs / 128       # linear lr scaling if batch had to shrink
        p = sp.Popen([sys.executable, "-u", "/content/src_run/run_experiment.py", "--config", cfg, "--data-root", DATA,
                      "--num-workers", "2", "--batch-size", str(bs), "--lr", str(lr)],
                     cwd=PROJ, stdout=sp.PIPE, stderr=sp.STDOUT, text=True, bufsize=1)
        oom = False
        for l in p.stdout:
            print(l, end=""); oom |= "out of memory" in l.lower()
        if p.wait() == 0: return True
        if oom and bs > 4: BS[model] = bs // 2; print(f"\n>>> OOM at batch {bs}; retrying with {bs // 2}\n")
        else: print(f"\n>>> run failed (attempt {attempt + 1}); retrying (resumes from checkpoint)\n"); time.sleep(15)
        torch.cuda.empty_cache()
    return False

for model in MODELS:
    for rate in RATES:
        cfg = f"configs/experiments/imagenet_subset_{model}_backdoor_{rate}.yaml"
        run_id = f"imagenet_subset_{model}_backdoor_{str(yaml.safe_load(open(f'{PROJ}/{cfg}'))['rate']).replace('.', 'p')}"
        res, t0 = Path(PROJ, "results", f"{run_id}_result.json"), time.time()
        print(f"\n{'=' * 70}\n{run_id}\n{'=' * 70}")
        if not res.is_file() and not train(model, cfg): status[run_id] = "TRAIN FAILED"; continue
        if not res.is_file(): status[run_id] = "NO RESULT FILE"; continue
        dest = push(run_id)
        status[run_id] = f"pushed -> {dest}" if dest else "TRAINED BUT PUSH FAILED (result still in runtime)"
        for f in glob.glob(f"{PROJ}/checkpoints/{run_id}*"): os.remove(f)
        print(f">>> {run_id}: {status[run_id]} ({(time.time() - t0) / 60:.1f} min)")

print("\n" + "=" * 70 + f"\nFINISHED in {(time.time() - t_all) / 3600:.2f} h on branch {BR}")
for k, v in status.items(): print(f"  {k}: {v}")
if AUTO_DISCONNECT and all(v.startswith("pushed") for v in status.values()):
    from google.colab import runtime; runtime.unassign()


imagenet_subset_resnet50_backdoor_0p001
[run] imagenet_subset_resnet50_backdoor_0p001
[poison] backdoor rate=0.001 n_poisoned=9
[epoch 1/40] train_acc=0.1070 clean_test_acc=0.1172 ASR=0.0000
[epoch 2/40] train_acc=0.1373 clean_test_acc=0.1279 ASR=0.0020
[epoch 3/40] train_acc=0.1446 clean_test_acc=0.1282 ASR=0.0000
[epoch 4/40] train_acc=0.1528 clean_test_acc=0.1753 ASR=0.0000
[epoch 5/40] train_acc=0.2057 clean_test_acc=0.2130 ASR=0.0000
[epoch 6/40] train_acc=0.2591 clean_test_acc=0.2673 ASR=0.1933
[epoch 7/40] train_acc=0.2964 clean_test_acc=0.2930 ASR=0.2629
[epoch 8/40] train_acc=0.3162 clean_test_acc=0.2910 ASR=0.2603
[epoch 9/40] train_acc=0.3449 clean_test_acc=0.3455 ASR=0.1588
[epoch 10/40] train_acc=0.3797 clean_test_acc=0.3720 ASR=0.1916
[epoch 11/40] train_acc=0.4145 clean_test_acc=0.4349 ASR=0.1215
[epoch 12/40] train_acc=0.4501 clean_test_acc=0.4601 ASR=0.0201
[epoch 13/40] train_acc=0.4795 clean_test_acc=0.4780 ASR=0.0387
[epoch 14/40] train_acc=0.5131 clean_test_acc=0.

In [ ]:
print(g("log", "--oneline", "-12")[1], g("status", "--short")[1], sorted(p.name for p in Path(PROJ, "results").glob("imagenet_subset_*")), sep="\n")